# Beloit College policy assistant

This notebook walks through the Vertex version of our RAG pipeline. It uses the same functions as `rag_pipeline.py`, so the notebook and terminal version stay consistent. The instructor's original `pipeline.py` is kept as a reference.

Run from the project folder with the `.venv` kernel after installing `requirements.txt` and signing in to Google Cloud. Setup instructions are in the README. Vertex calls incur charges; this notebook does not use an OpenAI key.


In [1]:
from pathlib import Path
import json
import rag_pipeline as rag
from evaluate import validate_gold, run_evaluation
assert (Path.cwd() / "corpus").is_dir(), "Open from the repository root"
RUN_GENERATION = True
RUN_EVALUATION = False  # True runs/resumes both variants over the 25 gold questions.
print("Project:", rag.PROJECT)
print("Models:", rag.EMBED_MODEL, rag.CHAT_MODEL)
print("Chunks:", rag.CHUNK_WORDS, "words, overlap", rag.OVERLAP_WORDS)


Project: dsda310-project2
Models: text-embedding-005 gemini-3.5-flash
Chunks: 250 words, overlap 50


## 1. Read the documents and split them into chunks

The supplied extractor has already saved the text in `corpus/`. We read those files, separate the metadata, convert PDF ligatures to ordinary letters, and remove page markers and standalone footer numbers before searching.

The baseline uses chunks of up to 250 words with 50 words of overlap. Overlap shares some context between neighboring chunks, but a long sentence can still be split. Each chunk keeps an ID such as `handbook-lockout#0` so we can find its source.

The smaller-chunk version uses 200 words with the same overlap. The next cell prints the counts for both. Some page-based extracts also contain neighboring policies, which can make retrieval less precise.


In [2]:
documents = rag.load_documents()
texts, sources = rag.build_chunks(documents)
print(len(documents), "documents;", len(texts), "chunks")
assert len(texts) >= 150
print(sources[0], texts[0][:300])

smaller_texts, _ = rag.build_chunks(documents, size=200, overlap=50)
print("200-word candidate:", len(smaller_texts), "chunks; both conditions retrieve 3")


23 documents; 179 chunks
catalog-academic-residency#0 Academic Residencies The Scott Crom Visiting Philosopher Program brings distinguished and infuential philosophers to campus for two days of talks, discussions, and classroom visits. Scott Crom was a beloved professor of philosophy at Beloit from 1954 to 1993. The series, established in 2010, is made
200-word candidate: 235 chunks; both conditions retrieve 3


## 2. Turn the chunks into embeddings

An embedding represents text as a vector. We use Vertex `text-embedding-005`, with RETRIEVAL_DOCUMENT for the chunks and RETRIEVAL_QUERY for questions.

The vectors are cached locally. If the text or embedding settings change, the cache is rebuilt. Automatic truncation is turned off so an oversized input gives an error instead of silently losing text.


In [3]:
embed_client, chat_client = rag.get_clients()
matrix = rag.load_or_embed(embed_client, texts)
print("Embedding matrix:", matrix.shape)


chunks or embedding model changed since the cache was written; re-embedding
embedding 179 chunks


  embedded 20/179


  embedded 40/179


  embedded 60/179


  embedded 80/179


  embedded 100/179


  embedded 120/179


  embedded 140/179


  embedded 160/179


  embedded 179/179
Embedding matrix: (179, 768)


## 3. Find relevant passages

We compare the question vector with the chunk vectors using cosine similarity and take the three highest-ranked chunks. A high score means similar text, not proof that a passage answers the question.

Both experiment versions retrieve three chunks. The difference is how the documents were split: 250 words in the baseline and 200 in the other version, both with 50-word overlap.


In [4]:
question = "Who should I contact if I am locked out of my room on Saturday night?"
hits = rag.retrieve(embed_client, matrix, texts, sources, question, k=3)
for passage, source, score in hits:
    print(source, round(score, 4), passage[:250])


handbook-lockout#2 0.7048 to carry room and building keys (and IDs) with them at all times. Locked doors and attention to keys are essential to students’ personal safety and the safety of their property. In the event a student is locked out of their room during a college work
handbook-student-ID#0 0.6523 Lost ID/Meal Cards/Keys/Key Card: Students who have lost their ID card/meal card/key card should report it to Residence Life and fill out a lost ID card form. Residence Life will make you a new card for free if it is your first time losing it. Additi
handbook-quiet-hours#0 0.6523 Lost ID/Meal Cards/Keys/Key Card: Students who have lost their ID card/meal card/key card should report it to Residence Life and fill out a lost ID card form. Residence Life will make you a new card for free if it is your first time losing it. Additi


## 4. Write an answer and attach its evidence

Gemini sees the question and retrieved source sentences. It returns an answer, an answerability flag, and the IDs of the sentences it used. Python then copies those sentences directly into the output. This avoids asking the model to retype PDF text, which caused corrupted quotes earlier.

The code checks that each quote appears in its claimed source, ignoring only case and extra spaces. The prompt asks the model to refuse when the evidence is missing. A matching quote still does not prove that an answer is complete or correct; we score correctness separately.


In [5]:
if RUN_GENERATION:
    answer = rag.answer_from_hits(chat_client, question, hits)
    print(json.dumps(answer, indent=2, ensure_ascii=False))
else:
    print("Generation skipped")


{
  "question": "Who should I contact if I am locked out of my room on Saturday night?",
  "retrieved": [
    [
      "handbook-lockout#2",
      0.7048467993736267
    ],
    [
      "handbook-student-ID#0",
      0.6522940397262573
    ],
    [
      "handbook-quiet-hours#0",
      0.6522940397262573
    ]
  ],
  "answer": "If you are locked out of your room on the weekend, you should contact Security.",
  "answerable": true,
  "evidence": [
    {
      "evidence_id": "handbook-lockout#2:s3",
      "quote": "If it is after office hours or on the weekend, the student should contact Security, who can let them in.",
      "source": "handbook-lockout#2"
    }
  ],
  "quote_verified": true,
  "citation_status": "valid"
}


## 5. Compare the two chunk sizes

We use the same 25 questions for both versions, including four questions the public corpus cannot answer. Expected answers and evidence are saved for evaluation, but are not sent to the model.

Only chunk size changes: 250 words versus 200 words. Both retrieve three chunks and keep 50 words of overlap. The five metrics are hit rate, quote fidelity, correctness, correct refusals, and wrong refusals. A refusal on any answerable gold question counts as a wrong refusal, even if retrieval missed the evidence.

Evaluation results are saved under `analysis/vertex_runs/`, with settings and separate scoring sheets. Mai and Parker each score correctness before comparing results and discussing disagreements. Those scores stay pending until both reviews are complete. See `analysis/evaluation_set.md` for the definitions and `analysis/results.md` for the counts.


In [6]:
gold = validate_gold()
print("Answerable:", sum(q["answerable"] for q in gold))
if RUN_EVALUATION:
    rows, summary, output = run_evaluation()
    print(json.dumps(summary, indent=2))
    print("Saved to", output)
else:
    print("Full evaluation skipped; use python rag_pipeline.py --evaluate in Terminal.")


Validated 25 questions and all evidence excerpts against 23 documents.
Answerable: 21
Full evaluation skipped; use python rag_pipeline.py --evaluate in Terminal.


## 6. Look at the failures

The failure report is in `analysis/failure_taxonomy.md`. For a failed question, read the retrieved passages as well as the answer: did retrieval miss the rule, or did the answer leave something out?

Keep API and sign-in errors separate from answer failures. A refusal is useful when the corpus cannot answer the question, but still counts against the system when the gold question is answerable. Good wording and a real citation do not make up for a missing condition.
